# `mutate` — Reference
`mutate` creates or overwrites columns using clean keyword argument syntax (`col="expr"` or `col=callable`). Each entry is evaluated in order via pandas `eval()` — a plain formula per column, or a lambda when needed.
### Calling Styles
1. **Keyword arguments** (most Pythonic): `.pt.mutate(bmi="body_mass_g / bill_length_mm ** 2", mass_kg="body_mass_g / 1000")`
2. **Dictionary unpacking** (for new column names with spaces): `.pt.mutate(**{"body mass kg": "body_mass_g / 1000"})`
3. **External file specification**: `.pt.mutate("@features.txt")`
---


In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src'))
if not os.path.exists(_src): _src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import numpy as np
import pytae as pt

penguins = pt.sample_data['penguins']

## A single derived column

In [2]:
# Body mass index style ratio — clean assignment formula via kwargs
(
    penguins
    .pt.mutate(bmi='body_mass_g / bill_length_mm ** 2')
    .pt.select('species', 'body_mass_g', 'bill_length_mm', 'bmi')
    .sample(10)
)


,species,body_mass_g,bill_length_mm,bmi
216,Chinstrap,3400.0,43.5,1.796803
166,Chinstrap,3575.0,45.9,1.696878
170,Chinstrap,3450.0,46.4,1.602445
44,Adelie,3000.0,37.0,2.191381
319,Gentoo,5250.0,51.1,2.010562
126,Adelie,3275.0,38.8,2.175444
118,Adelie,3350.0,35.7,2.628502
75,Adelie,4250.0,42.8,2.320072
183,Chinstrap,4300.0,54.2,1.463760
313,Gentoo,5650.0,49.5,2.305887


In [3]:
# Column names inside the expression must stay unquoted — quoting one turns it into
# a string literal, which will break arithmetic
try:
    penguins.pt.mutate(bmi="'body_mass_g' / 'bill_length_mm' ** 2")
except TypeError as exc:
    print(f"TypeError (as expected): {exc}")


TypeError (as expected): unsupported operand type(s) for ** or pow(): 'str' and 'int'


## Multiple entries in one call, and chaining a later entry off an earlier one
Entries are applied left to right, so a later expression can reference a column derived earlier in the *same* `mutate()` call.

In [4]:
# Two independent derived columns in one call via kwargs
(
    penguins
    .pt.mutate(heavy='body_mass_g > 4000', mass_kg='body_mass_g / 1000')
    .pt.select('species', 'body_mass_g', 'mass_kg', 'heavy')
    .sample(10)
)


,species,body_mass_g,mass_kg,heavy
268,Gentoo,5100.0,5.10,True
263,Gentoo,4750.0,4.75,True
185,Chinstrap,4100.0,4.10,True
285,Gentoo,5700.0,5.70,True
42,Adelie,3100.0,3.10,False
273,Gentoo,5000.0,5.00,True
168,Chinstrap,3300.0,3.30,False
281,Gentoo,5300.0,5.30,True
22,Adelie,3800.0,3.80,False
270,Gentoo,4850.0,4.85,True


In [5]:
# mass_lb references mass_kg, derived by the keyword just before it
(
    penguins
    .pt.mutate(mass_kg='body_mass_g / 1000', mass_lb='mass_kg * 2.20462')
    .pt.select('species', 'body_mass_g', 'mass_kg', 'mass_lb')
    .sample(10)
)


,species,body_mass_g,mass_kg,mass_lb
291,Gentoo,5000.0,5.00,11.023100
248,Gentoo,4600.0,4.60,10.141252
236,Gentoo,4150.0,4.15,9.149173
124,Adelie,3050.0,3.05,6.724091
265,Gentoo,4900.0,4.90,10.802638
98,Adelie,2900.0,2.90,6.393398
281,Gentoo,5300.0,5.30,11.684486
177,Chinstrap,4150.0,4.15,9.149173
233,Gentoo,5850.0,5.85,12.897027
60,Adelie,3150.0,3.15,6.944553


## String comparisons and local variables
String literals *inside* the expression (e.g. `'Adelie'`) need real quotes — column names stay bare. A variable from the calling scope can be referenced with an `@` prefix, same as pandas' own `eval()`/`query()`.


In [6]:
# String comparisons inside the expression formula
(
    penguins
    .pt.mutate(is_adelie="species == 'Adelie'")
    .pt.select('species', 'is_adelie')
    .sample(10)
)


,species,is_adelie
289,Gentoo,False
6,Adelie,True
288,Gentoo,False
238,Gentoo,False
248,Gentoo,False
340,Gentoo,False
83,Adelie,True
317,Gentoo,False
329,Gentoo,False
295,Gentoo,False


In [7]:
# @-prefixed names resolve against the scope that called mutate(), not the module
# — works identically whether calling pt.mutate() or chaining .pt.mutate()
threshold = 4000

(
    penguins
    .pt.mutate(heavy='body_mass_g >= @threshold')
    .pt.select('species', 'body_mass_g', 'heavy')
    .sample(10)
)


,species,body_mass_g,heavy
296,Gentoo,4600.0,True
192,Chinstrap,3950.0,False
115,Adelie,4075.0,True
80,Adelie,3200.0,False
5,Adelie,3650.0,False
183,Chinstrap,4300.0,True
220,Gentoo,4500.0,True
11,Adelie,3700.0,False
22,Adelie,3800.0,False
119,Adelie,3325.0,False


## Overwriting an existing column

In [8]:
# mutate() can overwrite a column in place, e.g. converting units
(
    penguins
    .pt.mutate(body_mass_g='body_mass_g / 1000')
    .pt.select('species', 'body_mass_g')
    .sample(10)
)


,species,body_mass_g
135,Adelie,3.90
220,Gentoo,4.50
40,Adelie,3.15
152,Chinstrap,3.50
273,Gentoo,5.00
303,Gentoo,5.35
241,Gentoo,5.00
45,Adelie,4.60
233,Gentoo,5.85
205,Chinstrap,4.05


## Column names with spaces — bracket syntax
Use square brackets `[col name]` to reference column names containing spaces without shell or syntax hazards (backticks are also supported):

In [9]:
import pandas as pd
spaced = pd.DataFrame({'body mass g': [3750, 4200], 'bill length mm': [39.1, 46.5]})

# Square brackets protect column names that contain spaces
spaced.pt.mutate(bmi="[body mass g] / [bill length mm] ** 2")


,body mass g,bill length mm,bmi
0,3750,39.1,2.452888
1,4200,46.5,1.942421


## Real-world pipeline: mutate → filter → select
`mutate()` chains like any other pytae method — filter on a column you just derived with `qry()`.

In [10]:
(
    penguins
    .pt.mutate(bmi='body_mass_g / bill_length_mm ** 2')
    .pt.qry(bmi='> 2')
    .pt.select('species', 'island', 'body_mass_g', 'bill_length_mm', 'bmi')
    .sample(10)
)


,species,island,body_mass_g,bill_length_mm,bmi
276,Gentoo,Biscoe,4300.0,43.8,2.241404
97,Adelie,Dream,4350.0,40.3,2.678423
304,Gentoo,Biscoe,4750.0,44.9,2.356139
228,Gentoo,Biscoe,4400.0,43.3,2.346804
78,Adelie,Torgersen,3550.0,36.2,2.709014
110,Adelie,Biscoe,3825.0,38.1,2.635005
305,Gentoo,Biscoe,5600.0,50.8,2.170004
72,Adelie,Torgersen,3550.0,39.6,2.263800
288,Gentoo,Biscoe,4700.0,43.5,2.483816
274,Gentoo,Biscoe,4900.0,46.5,2.266158


## Conditional column creation — `if_else()`, `case_when()`, `map()`
Plain `eval()` has no ternary/`where()` support, so `mutate()` provides three dplyr-style helpers as ordinary function calls, evaluated via `np.where()`/`np.select()`/`Series.map()`: `if_else(condition, true_value, false_value)`, `case_when((cond1, val1), (cond2, val2), ..., default)`, and `map(column, {key: value, ...}, default)`. A trailing bare argument to `case_when` is the catch-all default (like SQL ELSE). Because they are ordinary calls, they compose and chain with each other and with any pandas method. String outcomes need quotes; conditions are vectorized — prefer `and`/`or`/`not` (the bitwise `&`/`|`/`~` also work).

In [11]:
# if_else(condition, true_value, false_value) — like dplyr's if_else()
# true_value and false_value can be scalars or column names; conditions are vectorized
(
    penguins
    .pt.mutate(weight_class="if_else(body_mass_g > 4000, 'heavy', 'light')")
    .pt.select('species', 'body_mass_g', 'weight_class')
    .sample(10)
)


,species,body_mass_g,weight_class
100,Adelie,3725.0,light
294,Gentoo,4700.0,heavy
165,Chinstrap,4050.0,heavy
136,Adelie,3175.0,light
17,Adelie,4500.0,heavy
316,Gentoo,4925.0,heavy
3,Adelie,NaN,light
186,Chinstrap,3600.0,light
2,Adelie,3250.0,light
305,Gentoo,5600.0,heavy


In [12]:
# case_when supports tuples or flat pairs with default=
(
    penguins
    .pt.mutate(
        size_class="case_when(body_mass_g >= 4500, 'large', body_mass_g >= 3500, 'medium', default='small')"
    )
    .pt.select('species', 'body_mass_g', 'size_class')
    .sample(10)
)


,species,body_mass_g,size_class
44,Adelie,3000.0,small
111,Adelie,4600.0,large
36,Adelie,3950.0,medium
318,Gentoo,4625.0,large
236,Gentoo,4150.0,medium
262,Gentoo,4300.0,medium
67,Adelie,4100.0,medium
123,Adelie,3875.0,medium
93,Adelie,4450.0,medium
266,Gentoo,4200.0,medium


In [13]:
# map(column, {key: value, ...}, default) — recode a column through a dictionary lookup
# unmapped keys become default if given, else NaN
(
    penguins
    .pt.mutate(
        island_code="map(island, {'Torgersen': 'TOR', 'Biscoe': 'BIS'}, 'OTH')"
    )
    .pt.select('species', 'island', 'island_code')
    .sample(10)
)


,species,island,island_code
207,Chinstrap,Dream,OTH
169,Chinstrap,Dream,OTH
272,Gentoo,Biscoe,BIS
71,Adelie,Torgersen,TOR
232,Gentoo,Biscoe,BIS
37,Adelie,Dream,OTH
94,Adelie,Dream,OTH
270,Gentoo,Biscoe,BIS
287,Gentoo,Biscoe,BIS
73,Adelie,Torgersen,TOR


## First non-null resolution — `coalesce()`

`coalesce(col1, col2, ..., default)` evaluates candidates left-to-right and returns the first non-null value per row, like SQL `COALESCE()` or `dplyr::coalesce()`:

In [14]:
contacts = pd.DataFrame({
    'mobile': [None, '555-1234', None],
    'home': ['555-5678', None, None],
    'work': [None, None, '555-9012'],
})

contacts.pt.mutate(preferred_contact="coalesce(mobile, home, work, 'N/A')")


,mobile,home,work,preferred_contact
0,NaN,555-5678,NaN,555-5678
1,555-1234,NaN,NaN,555-1234
2,NaN,NaN,555-9012,555-9012


## Combining expressions and callables in kwargs

`mutate()` cleanly mixes expression strings and python callables (e.g. lambdas) in keyword arguments:


In [15]:
(
    penguins
    .pt.mutate(
        bmi='body_mass_g / bill_length_mm ** 2',
        mass_kg='body_mass_g / 1000',
        is_heavy=lambda df: df['body_mass_g'] > 4000,
    )
    .pt.select('species', 'bmi', 'mass_kg', 'is_heavy')
    .head(5)
)


,species,bmi,mass_kg,is_heavy
0,Adelie,2.452888,3.75,False
1,Adelie,2.435507,3.80,False
2,Adelie,2.001121,3.25,False
3,Adelie,NaN,NaN,False
4,Adelie,2.561456,3.45,False


## Loading specs from an external file — `@specs.txt`
For complex feature engineering or shared pipelines across Python and the CLI, specs can be loaded from an external file. The file supports multiline expressions, `#` comments, empty lines, and column chaining (where later formulas reference columns defined earlier in the same file).
### File content of `penguin_features.txt`:
```text
# Engineering features for penguins
mass_kg = body_mass_g / 1000
# Convert kg to lbs (references mass_kg computed above)
mass_lb = mass_kg * 2.20462
```


In [16]:
# Write a small demo spec file with comments and assignment '=' syntax
spec_content = """# Engineering features for penguins
mass_kg = body_mass_g / 1000

# Convert kg to lbs
mass_lb = mass_kg * 2.20462
"""
with open("penguin_features.txt", "w") as f:
    f.write(spec_content)

# Load directly using @filename
result = penguins.pt.mutate("@penguin_features.txt")
out = (
    result
    .pt.select("species", "mass_kg", "mass_lb")
    .sample(5)
)

# Clean up demo file
import os
os.remove("penguin_features.txt")

out


,species,mass_kg,mass_lb
186,Chinstrap,3.60,7.936632
212,Chinstrap,3.95,8.708249
149,Adelie,3.75,8.267325
107,Adelie,3.90,8.598018
281,Gentoo,5.30,11.684486


## Columns with spaces — SQL-style brackets `[col]`

SQL-style square brackets `[col a]` are supported alongside backticks, avoiding shell backtick substitution hazards:

In [17]:
spaced.pt.mutate(ratio='[body mass g] / [bill length mm]')


,body mass g,bill length mm,ratio
0,3750,39.1,95.907928
1,4200,46.5,90.322581


## Creating new columns with spaces — string expressions `[col] = ...`

You can create columns with spaces directly using string assignment expressions with brackets `[new col] = ...` with zero dictionary unpacking.

(Dictionary unpacking `**{'col name': ...}` is also supported for kwargs).

In [18]:
# Create a new column with spaces using bracket string expression (no dict unpacking!)
(
    spaced
    .pt.mutate("[body mass ratio] = [body mass g] / [bill length mm]")
    .pt.select('[body mass g]', '[body mass ratio]')
    .head()
)

,body mass g,body mass ratio
0,3750,95.907928
1,4200,90.322581


In [19]:
# Define multiple new columns with spaces at once via string expressions
(
    penguins
    .pt.mutate(
        "[body mass kg] = body_mass_g / 1000",
        "[bill ratio] = bill_length_mm / bill_depth_mm",
    )
    .pt.select('species', '[body mass kg]', '[bill ratio]')
    .head()
)

,species,body mass kg,bill ratio
0,Adelie,3.75,2.090909
1,Adelie,3.80,2.270115
2,Adelie,3.25,2.238889
3,Adelie,NaN,NaN
4,Adelie,3.45,1.901554


Note: When loading specs from an external file (`@specs.txt`) or via the CLI (`-mutate`), names with spaces can be written directly on the left-hand side of `=` without dictionary unpacking:

```text
# inside an external spec file or CLI flag:
my bmi = [body mass g] / [bill length mm] ** 2
```


## Grouped mutations — `mutate(..., by=...)`

Unlike `pt.agg()` which collapses rows ($N \to K$), `mutate(..., by=...)` evaluates window calculations and group-level summaries and broadcasts them back to every row ($N \to N$).

Supported features:
- **Injected aggregation functions**: `mean(x)`, `sum(x)`, `median(x)`, `min(x)`, `max(x)`, `std(x)`, `var(x)`
- **Group row count**: `n` or `n()` (dplyr / Polars convention)
- **Sequential expressions**: Later expressions within the same `mutate` call can immediately reference earlier group-derived columns (e.g. `diff = tip - avg_tip`)
- **Arbitrary output column names**: Name target columns cleanly without hardcoded prefixes
- **Multi-column grouping**: `by=['species', 'island']` or `by='species, island'`
- **Spaced column names**: `[total bill]`
- **Missing group handling**: `dropna=False` retains NA groups


In [20]:
# Compute group average tip, deviation from group average, and group sample size
tips = pt.sample("tips")
(
    tips
    .pt.select("day", "time", "total_bill", "tip")
    .pt.mutate(
        "avg_tip = mean(tip), diff = tip - avg_tip, group_size = n",
        by="day",
    )
    .head(6)
)

,day,time,total_bill,tip,avg_tip,diff,group_size
0,Sun,Dinner,16.99,1.01,3.255132,-2.245132,76
1,Sun,Dinner,10.34,1.66,3.255132,-1.595132,76
2,Sun,Dinner,21.01,3.50,3.255132,0.244868,76
3,Sun,Dinner,23.68,3.31,3.255132,0.054868,76
4,Sun,Dinner,24.59,3.61,3.255132,0.354868,76
5,Sun,Dinner,25.29,4.71,3.255132,1.454868,76


In [21]:
# Multi-column grouping with spaced columns
(
    tips
    .rename(columns={"total_bill": "total bill"})
    .pt.select("day", "time", "total bill", "tip")
    .pt.mutate(
        "[avg bill] = mean([total bill]), [bill diff] = [total bill] - [avg bill], count = n()",
        by=["day", "time"],
    )
    .head(6)
)

,day,time,total bill,tip,avg bill,bill diff,count
0,Sun,Dinner,16.99,1.01,21.41,-4.42,76
1,Sun,Dinner,10.34,1.66,21.41,-11.07,76
2,Sun,Dinner,21.01,3.50,21.41,-0.40,76
3,Sun,Dinner,23.68,3.31,21.41,2.27,76
4,Sun,Dinner,24.59,3.61,21.41,3.18,76
5,Sun,Dinner,25.29,4.71,21.41,3.88,76


In [22]:
# De-meaning and z-score calculation per group
(
    penguins
    .pt.select("species", "body_mass_g")
    .pt.mutate(
        "mean_mass = mean(body_mass_g), std_mass = std(body_mass_g), z_score = (body_mass_g - mean_mass) / std_mass",
        by="species",
    )
    .head(6)
)

,species,body_mass_g,mean_mass,std_mass,z_score
0,Adelie,3750.0,3700.662252,458.566126,0.107591
1,Adelie,3800.0,3700.662252,458.566126,0.216627
2,Adelie,3250.0,3700.662252,458.566126,-0.982764
3,Adelie,NaN,3700.662252,458.566126,NaN
4,Adelie,3450.0,3700.662252,458.566126,-0.546622
5,Adelie,3650.0,3700.662252,458.566126,-0.110480


### Handling missing grouping values with `dropna`

By default, `dropna=False` computes group statistics across `NA` rows as their own distinct group (no silent data loss). Pass `dropna=True` to exclude rows with `NA` in grouping columns from calculations, filling `NaN` in mutated columns while preserving total row count and index alignment:


In [23]:
sample_data = pd.DataFrame({
    "grp": ["A", "A", None, "B", None],
    "val": [10.0, 20.0, 100.0, 30.0, 200.0]
})

# dropna=False (default): NA rows form a distinct group
print("dropna=False (default):")
print(sample_data.pt.mutate("avg = mean(val), sz = n", by="grp"))

# dropna=True: NA groups receive NaN
print("\ndropna=True:")
print(sample_data.pt.mutate("avg = mean(val), sz = n", by="grp", dropna=True))


dropna=False (default):
   grp    val    avg  sz
0    A   10.0   15.0   2
1    A   20.0   15.0   2
2  NaN  100.0  150.0   2
3    B   30.0   30.0   1
4  NaN  200.0  150.0   2

dropna=True:
   grp    val   avg   sz
0    A   10.0  15.0  2.0
1    A   20.0  15.0  2.0
2  NaN  100.0   NaN  NaN
3    B   30.0  30.0  1.0
4  NaN  200.0   NaN  NaN
